In [ ]:
!pip install langchain_community -q

In [ ]:
!pip install langchain-experimental -q

In [ ]:
!pip install langchain-openai -q

In [ ]:
import os
import base64
import mimetypes
import requests

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ------------------------------------------------------------
# LM Studio local server (OpenAI-compatible)
# ------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = "lm-studio"                
os.environ["OPENAI_API_BASE"] = "http://localhost:1234/v1"

MODEL = "qwen/qwen3-vl-4b" 

# ------------------------------------------------------------
# Helpers: ALWAYS return a base64 data URL
# ------------------------------------------------------------
def _file_to_data_url(path: str) -> str:
    mime, _ = mimetypes.guess_type(path)
    if mime is None:
        mime = "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def _fetch_url_to_data_url(url: str, timeout: int = 30) -> str:

    r = requests.get(url, timeout=timeout)
    r.raise_for_status()

    mime = r.headers.get("Content-Type", "image/jpeg")
    if not mime.startswith("image/"):
        mime = "image/jpeg"
    b64 = base64.b64encode(r.content).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def _to_data_url(path_or_url: str) -> str:
    if path_or_url.startswith(("http://", "https://")):
        return _fetch_url_to_data_url(path_or_url)
    elif path_or_url.startswith("data:"):
        return path_or_url  
    else:
        return _file_to_data_url(path_or_url)

def _image_part_always_base64(path_or_url: str) -> dict:
    data_url = _to_data_url(path_or_url)

    return {"type": "image_url", "image_url": {"url": data_url}}

# ------------------------------------------------------------
# Initialize ChatOpenAI client
# ------------------------------------------------------------
llm = ChatOpenAI(
    model=MODEL,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    temperature=0.2,
    max_tokens=256,
)

# ------------------------------------------------------------
# Visual Q&A
# ------------------------------------------------------------
def vqa(question: str, image_path_or_url: str) -> str:
    messages = [
        SystemMessage(content="You are a precise visual QA assistant. Answer strictly based on the image."),
        HumanMessage(content=[
            {"type": "text", "text": question},
            _image_part_always_base64(image_path_or_url),
        ])
    ]
    resp = llm.invoke(messages) 
    return resp.content

# ------------------------------------------------------------
# Example usage
# ------------------------------------------------------------
# 1) Public image URL (now converted to base64 automatically)
print("URL →", vqa(
    "What is the main object and what is happening?",
    "https://images.unsplash.com/photo-1500530855697-b586d89ba3ee?w=1200"
))

# 2) Local image
# print("LOCAL →", vqa("Describe the scene briefly.", "my_image.jpg"))


In [ ]:
import os
import base64
import mimetypes
import requests
from typing import List  

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ------------------------------------------------------------
# LM Studio local server (OpenAI-compatible)
# ------------------------------------------------------------
os.environ["OPENAI_API_KEY"]  = "lm-studio"               
os.environ["OPENAI_API_BASE"] = "http://localhost:1234/v1" 


MODEL = "qwen/qwen3-vl-4b"

# ------------------------------------------------------------
# Base64 helpers (ALWAYS return data URLs to satisfy some LM Studio builds)
# ------------------------------------------------------------
def _file_to_data_url(path: str) -> str:
    mime, _ = mimetypes.guess_type(path)
    mime = mime or "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def _fetch_url_to_data_url(url: str, timeout: int = 30) -> str:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    mime = r.headers.get("Content-Type", "image/jpeg")
    if not (isinstance(mime, str) and mime.startswith("image/")):
        mime = "image/jpeg"
    b64 = base64.b64encode(r.content).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def _to_data_url(path_or_url: str) -> str:
    if path_or_url.startswith(("http://", "https://")):
        return _fetch_url_to_data_url(path_or_url)
    elif path_or_url.startswith("data:"):
        return path_or_url
    else:
        return _file_to_data_url(path_or_url)

def _image_part(path_or_url: str) -> dict:
    return {"type": "image_url", "image_url": {"url": _to_data_url(path_or_url)}}

# ------------------------------------------------------------
# LLM client
# ------------------------------------------------------------
llm = ChatOpenAI(
    model=MODEL,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    openai_api_base=os.environ["OPENAI_API_BASE"],
    temperature=0.2,
    max_tokens=512,
)

# ------------------------------------------------------------
# Multi-image reasoning
# ------------------------------------------------------------
def vqa_multi_image(question: str, images: List[str]) -> str:
    """
    Compare/reason over multiple images in one call.
    images: list of local paths or http(s) URLs.
    """
    if not images:
        raise ValueError("Provide at least one image path or URL.")

    content = [{"type": "text", "text": question}]
    for img in images:
        content.append(_image_part(img))

    messages = [
        SystemMessage(content="You are a precise visual QA assistant. Compare and reason over ALL provided images."),
        HumanMessage(content=content),
    ]
    resp = llm.invoke(messages) 
    return resp.content


if __name__ == "__main__":

    ans = vqa_multi_image(
        "Compare the two scenes. List three major differences in weather, layout, or objects.",
        [
            "https://images.unsplash.com/photo-1519681393784-d120267933ba?w=1200",
            "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=1200",
        ],
    )
    print("Multi-image (differences) →", ans)
